In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Transactions.csv")
df.head()

,Transaction_ID,CustomerId,Transaction_Date,Transaction_Type,Amount
0,7423388,15634602,2023-09-28,LoanPayment,3909.47
1,3234489,15634602,2023-08-03,LoanPayment,2323.28
2,8204212,15634602,2023-04-10,Fee,3554.96
3,5523669,15634602,2023-11-05,Withdrawal,3623.89
4,5521373,15634602,2023-10-21,Withdrawal,950.03


In [7]:
summary = {
    "Shape": df.shape,
    "Colums":list(df.columns),
    "Missing Values":df.isnull().sum().to_dict(),
    "Duplicate Rows":int(df.duplicated().sum())
}

counts = df['Transaction_Type'].value_counts().to_dict()
invalid_types = df[~df['Transaction_Type'].isin(["Deposit","Withdrawal","LoanPayment","Fee"])]
invalid_dates = df[~pd.to_datetime(df['Transaction_Date'], errors = 'coerce').notna()]

negative_or_zero = df[df['Amount']<=0]

q1 = df['Amount'].quantile(0.25)
q3 = df['Amount'].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 -1.5*iqr
upper_limit = q3+1.5*iqr

outliers = df[(df['Amount']<lower_limit)|(df['Amount']>upper_limit)]

print("Summary:", summary)
print("\nTransaction_Type_Counts:", counts)
print("\nInvalid_Transaction_Type:", invalid_types)
print("\nInvalid_dates:", invalid_dates)
print("\nOutliers_Per_Column:", len(outliers))
print("\nNegative_or_zero:", len(negative_or_zero))

Summary: {'Shape': (120804, 5), 'Colums': ['Transaction_ID', 'CustomerId', 'Transaction_Date', 'Transaction_Type', 'Amount'], 'Missing Values': {'Transaction_ID': 0, 'CustomerId': 0, 'Transaction_Date': 0, 'Transaction_Type': 0, 'Amount': 0}, 'Duplicate Rows': 0}

Transaction_Type_Counts: {'Deposit': 30367, 'Fee': 30264, 'Withdrawal': 30102, 'LoanPayment': 30071}

Invalid_Transaction_Type: Empty DataFrame
Columns: [Transaction_ID, CustomerId, Transaction_Date, Transaction_Type, Amount]
Index: []

Invalid_dates: Empty DataFrame
Columns: [Transaction_ID, CustomerId, Transaction_Date, Transaction_Type, Amount]
Index: []

Outliers_Per_Column: 0

Negative_or_zero: 0


In [11]:
txn_per_cust = df.groupby('CustomerId')['Transaction_ID'].count().reset_index()
txn_per_cust.columns = ['CustomerId', 'Transaction_Count']


In [10]:
df.groupby('Transaction_Type')['Amount'].sum()

Transaction_Type
Deposit        76603863.83
Fee            76398764.35
LoanPayment    75546485.50
Withdrawal     75926417.00
Name: Amount, dtype: float64

In [9]:
df.head()

,Transaction_ID,CustomerId,Transaction_Date,Transaction_Type,Amount,Month
0,7423388,15634602,2023-09-28,LoanPayment,3909.47,2023-09
1,3234489,15634602,2023-08-03,LoanPayment,2323.28,2023-08
2,8204212,15634602,2023-04-10,Fee,3554.96,2023-04
3,5523669,15634602,2023-11-05,Withdrawal,3623.89,2023-11
4,5521373,15634602,2023-10-21,Withdrawal,950.03,2023-10


In [12]:

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Month'] = df['Transaction_Date'].dt.to_period('M')
monthly_trend = df.groupby(['Month', 'Transaction_Type'])['Amount'].sum().unstack()


In [13]:
customers = pd.read_csv("Customers_Cleaned.csv")
merged = customers.merge(txn_per_cust, on='CustomerId', how='left')